# Notebook 03 — Error Analysis

**Purpose**: Understand where the model fails and design improvements.

This notebook:
1. Loads OOF predictions from a trained model
2. Shows per-date competition score over time
3. Identifies worst-performing symbols
4. Analyses residuals by feature buckets
5. Compares baseline vs model improvement

**Run after**: `make evaluate EXP=configs/experiments/exp001_lgbm_baseline.yaml`
**Data sources**: `artifacts/exp001_lgbm_baseline/oof_predictions.parquet`,
                  `reports/exp001_lgbm_baseline/`

In [ ]:
import polars as pl
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

import sys; sys.path.insert(0, str(Path('../src')))
from janestreet_forecasting.data import schemas as S
from janestreet_forecasting.modeling.metrics import (
    competition_score, weighted_r2, compute_all_metrics
)
from janestreet_forecasting.evaluation.diagnostics import compute_error_by_symbol
from janestreet_forecasting.paths import ARTIFACTS_DIR, REPORTS_DIR

plt.rcParams['figure.dpi'] = 120
sns.set_theme(style='whitegrid')

EXPERIMENT_ID = 'exp001_lgbm_baseline'
ARTIFACT_DIR = ARTIFACTS_DIR / EXPERIMENT_ID
REPORT_DIR = REPORTS_DIR / EXPERIMENT_ID

## 1. OOF Prediction Summary

In [ ]:
oof_path = ARTIFACT_DIR / 'oof_predictions.parquet'
if not oof_path.exists():
    print(f'OOF predictions not found at {oof_path}. Run `make train` first.')
    raise SystemExit

oof_df = pl.read_parquet(oof_path)
print(f'OOF predictions: {len(oof_df):,} rows')
print(f'Prediction range: [{oof_df["oof_prediction"].min():.4f}, {oof_df["oof_prediction"].max():.4f}]')
print(f'Target range: [{oof_df[S.TARGET_COL].min():.4f}, {oof_df[S.TARGET_COL].max():.4f}]')

# Overall metrics
metrics = compute_all_metrics(
    y_true=oof_df[S.TARGET_COL].to_numpy(),
    y_pred=oof_df['oof_prediction'].to_numpy(),
    weights=oof_df[S.WEIGHT].to_numpy(),
    dates=oof_df[S.DATE_ID].to_numpy(),
)
print('\n=== Overall OOF Metrics ===')
for k, v in metrics.items():
    print(f'  {k}: {v:.4f}')

## 2. Per-Date Competition Score

In [ ]:
dates = oof_df[S.DATE_ID].to_numpy()
y_true = oof_df[S.TARGET_COL].to_numpy()
y_pred = oof_df['oof_prediction'].to_numpy()
weights = oof_df[S.WEIGHT].to_numpy()

unique_dates = np.unique(dates)
daily_scores = []
for d in unique_dates:
    mask = dates == d
    score = competition_score(y_true[mask], y_pred[mask], weights[mask], dates[mask])
    daily_scores.append(score)

daily_scores = np.array(daily_scores)

fig, axes = plt.subplots(2, 1, figsize=(14, 8))

axes[0].plot(unique_dates, daily_scores, lw=0.7, alpha=0.8, color='steelblue')
window = min(30, len(daily_scores) // 3)
if window > 1:
    rolling = np.convolve(daily_scores, np.ones(window)/window, mode='valid')
    axes[0].plot(unique_dates[window-1:], rolling, lw=2, color='red', label=f'Rolling({window})')
axes[0].axhline(0, color='black', lw=0.5, ls='--')
axes[0].axhline(daily_scores.mean(), color='green', lw=1, ls='--', label=f'Mean={daily_scores.mean():.3f}')
axes[0].set_title('Per-Date Competition Score (OOF)')
axes[0].set_xlabel('date_id')
axes[0].legend()

axes[1].hist(daily_scores, bins=50, color='steelblue', alpha=0.8)
axes[1].axvline(0, color='red', ls='--', lw=1, label='zero')
axes[1].axvline(daily_scores.mean(), color='green', ls='--', lw=1.5, label=f'mean={daily_scores.mean():.3f}')
axes[1].set_title('Distribution of Per-Date Scores')
axes[1].set_xlabel('competition_score')
axes[1].legend()

plt.tight_layout()
plt.show()

negative_frac = (daily_scores < 0).mean()
print(f'Fraction of dates with negative score: {negative_frac:.1%}')

## 3. Worst Performing Symbols

In [ ]:
pred_df = oof_df.rename({'oof_prediction': 'y_pred', S.TARGET_COL: 'y_true'})
symbol_errors = compute_error_by_symbol(pred_df)

print('Top 10 worst symbols by RMSE:')
print(symbol_errors.head(10).to_pandas().to_string(index=False))

## 4. Prediction vs Actual Scatter

In [ ]:
# Sample for plotting speed
sample_idx = np.random.choice(len(oof_df), size=min(10_000, len(oof_df)), replace=False)
y_t = y_true[sample_idx]
y_p = y_pred[sample_idx]

fig, ax = plt.subplots(figsize=(8, 8))
ax.hexbin(y_t, y_p, gridsize=50, cmap='Blues', mincnt=1)
ax.plot([y_t.min(), y_t.max()], [y_t.min(), y_t.max()], 'r--', lw=1, label='perfect')
ax.set_xlabel('Actual (responder_6)')
ax.set_ylabel('Predicted')
ax.set_title(f'Prediction vs Actual | OOF R²={weighted_r2(y_true, y_pred, weights):.4f}')
ax.legend()
plt.tight_layout()
plt.show()